[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/lab04_place_equity.ipynb) · [Course home](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/00_course_home.ipynb)

**Google Colab version.** **Before you start: File → Save a copy in Drive.** Work only in your copy (its title starts with *Copy of*); changes to this original are not saved. Rename your copy `Lab 4 — <your last name>` (click the title at the top left).

**To submit:** **Runtime → Run all** and check that every cell ran, then **File → Save**. Click
**Share** (top right), add **steven.elliott@student.fairfield.edu** as **Editor**, and send. Colab keeps your saved versions
under **File → Revision history**; **File → Save and pin revision** keeps one permanently, which is how you
mark a submission.

**Coming back later?** Colab forgets variables and downloaded files when the runtime disconnects (after about 90 minutes idle). Open your copy and click **Runtime → Run all**.

# Lab 4 — Place, population, and equity
*Week 4. Baseline: sections 1–4. Powerups: section 5.*

Hospitals sit in communities. This week you attach county-level population health from **CDC PLACES**
to hospital outcomes from CMS, and chart one disparity. The hard part is not the chart; it is the join.

**How labs work.** Save a copy in Drive, work in your copy, share it back with the instructor
when you submit. Every lab has a **baseline** (everyone finishes it in the session) and
**powerups** (optional depth). AI assistants may write code for you; **you own every line** and
may be asked to explain any of it. Cells marked `# TODO` are yours to fill in.

In [ ]:
# --- Course setup: run this cell first -------------------------------------------
# Loaders for the public datasets this course uses. The canonical copy of this code
# is labs/course_data.py in the course repo; you will read it line by line in week 3.
import io, json, urllib.request, urllib.parse
from pathlib import Path
import pandas as pd
import duckdb

CACHE = Path("data"); CACHE.mkdir(exist_ok=True)

def sql(query: str) -> pd.DataFrame:
    """Run SQL over any DataFrame in this notebook by its variable name (DuckDB)."""
    return duckdb.sql(query).df()

def _get(url, params=None, timeout=120) -> bytes:
    if params:
        url = url + "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={"User-Agent": "healthcare-analytics-course/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- CMS Care Compare (data.cms.gov/provider-data) --------------------------------
# File names change at every CMS refresh; dataset identifiers do not. We ask the
# catalog for the current CSV URL every time instead of hard-coding a file name.
CMS_CATALOG = "https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items"
CMS_IDS = {
    "hospital_general_info": "xubh-q36u",   # one row per hospital
    "hrrp":                  "9n3s-kdb3",   # Hospital Readmissions Reduction Program, one row per hospital x measure
    "unplanned_visits":      "632h-zaca",   # readmissions / ED visits, one row per hospital x measure
    "complications_deaths":  "ynj2-r877",
    "timely_effective":      "yv7e-xc69",
}

def cms_csv_url(identifier: str) -> str:
    # Ask for the one dataset by id; the full keyword search can take minutes or time out.
    item = json.loads(_get(f"{CMS_CATALOG}/{identifier}", timeout=30))
    for d in item.get("distribution", []):
        if d.get("mediaType") == "text/csv":
            return d["downloadURL"]
    raise KeyError(f"No CSV distribution found for CMS dataset {identifier}")

# Course mirror: a snapshot of every CMS file the labs use, for when CMS is down or rate-limits
# a classroom downloading the same file at once. The live source is always tried first.
COURSE_DATA_BASE = "https://raw.githubusercontent.com/stevenelliottjr/healthcare-analytics-course-data/main/data"

def _fetch_or_mirror(path: Path, fetch, mirror_file: str) -> None:
    """Save `path` from the live source; if that fails, from the course mirror."""
    try:
        path.write_bytes(fetch())
    except Exception as e:
        print(f"Live download failed ({type(e).__name__}); using the course mirror for {path.name}.")
        path.write_bytes(_get(f"{COURSE_DATA_BASE}/{mirror_file}", timeout=300))

def load_cms(name: str, **read_csv_kwargs) -> pd.DataFrame:
    """Download (once) and read a Care Compare file. Everything is read as TEXT on purpose:
    IDs keep their leading zeros and 'Not Available' stays visible instead of silently
    becoming NaN. Converting columns to numbers is your job, and part of the lesson."""
    path = CACHE / f"cms_{name}.csv"
    if not path.exists():
        _fetch_or_mirror(path, lambda: _get(cms_csv_url(CMS_IDS[name]), timeout=300), f"cms/cms_{name}.csv")
    kw = dict(dtype=str, keep_default_na=False, encoding="utf-8-sig")
    kw.update(read_csv_kwargs)
    return pd.read_csv(path, **kw)

def to_num(s: pd.Series) -> pd.Series:
    """Text -> number; anything that is not a number ('Not Available', 'N/A', '') becomes NaN."""
    return pd.to_numeric(s, errors="coerce")

# ---- CDC PLACES, county release (Socrata API on data.cdc.gov) ---------------------
PLACES_COUNTY = "https://data.cdc.gov/resource/swc5-untb.json"
PLACES_SELECT = ("year,stateabbr,statedesc,locationname,locationid,categoryid,measureid,measure,"
                 "data_value,low_confidence_limit,high_confidence_limit,totalpopulation")

def load_places(measure_ids, data_value_type="Age-adjusted prevalence", state=None, page=50000) -> pd.DataFrame:
    """County-level model-based prevalence estimates (percent of adults). locationid is the
    5-digit county FIPS. Note: Connecticut reports *planning regions*, not counties, since 2022."""
    where = "measureid in(" + ",".join(f"'{m}'" for m in measure_ids) + f") AND data_value_type='{data_value_type}'"
    if state:
        where += f" AND stateabbr='{state}'"
    frames, offset = [], 0
    while True:
        chunk = pd.DataFrame(json.loads(_get(PLACES_COUNTY, {"$where": where, "$select": PLACES_SELECT,
                                                               "$limit": page, "$offset": offset})))
        frames.append(chunk)
        if len(chunk) < page:
            break
        offset += page
    df = pd.concat(frames, ignore_index=True)
    for c in ["data_value", "low_confidence_limit", "high_confidence_limit", "totalpopulation"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

# ---- Week 4 powerups (links checked 2026-10-26; 2022 is the latest SVI release) ----
SVI_COUNTY_URL = "https://svi.cdc.gov/Documents/Data/2022/csv/states_counties/SVI_2022_US_county.csv"
def load_svi_county() -> pd.DataFrame:
    """CDC/ATSDR Social Vulnerability Index 2022, one row per county.
    Key columns: FIPS, STATE, COUNTY, E_TOTPOP, RPL_THEMES (overall percentile 0-1),
    RPL_THEME1..RPL_THEME4. The value -999 means 'not available'."""
    path = CACHE / "svi_2022_county.csv"
    if not path.exists():
        path.write_bytes(_get(SVI_COUNTY_URL, timeout=300))
    return pd.read_csv(path, dtype={"FIPS": str}).replace(-999, pd.NA)

CHR_URL = "https://www.countyhealthrankings.org/sites/default/files/media/document/analytic_data2025_v3.csv"
def load_county_health_rankings() -> pd.DataFrame:
    """County Health Rankings analytic file. Row 0 holds readable labels and row 1 the variable
    codes; we skip the labels so the codes (fipscode, v001..v1xx) become the column names.
    The data dictionary on the CHR site maps each v-code to a measure."""
    path = CACHE / "chr_analytic_2025_v3.csv"
    if not path.exists():
        path.write_bytes(_get(CHR_URL, timeout=300))
    return pd.read_csv(path, skiprows=[0], dtype={"fipscode": str, "statecode": str, "countycode": str})

print("course setup ready")


## 1. County population health from CDC PLACES
PLACES gives model-based prevalence estimates for ~40 measures for every county (percent of adults).
We take four: diagnosed diabetes, obesity, adults 18–64 without insurance, routine checkup in the past year.

In [ ]:
MEASURES = ["DIABETES", "OBESITY", "ACCESS2", "CHECKUP"]
places = load_places(MEASURES)
print(places.shape, "| years:", sorted(places["year"].unique()))
county = (places.pivot_table(index=["stateabbr", "locationname", "locationid", "totalpopulation"],
                             columns="measureid", values="data_value").reset_index())
county.head()

## 2. Hospital outcomes: heart-failure readmission
From *Unplanned Hospital Visits*, keep the risk-standardized 30-day heart-failure readmission rate
(`READM_30_HF`, the same condition as week 2) and attach each hospital's `State` and `County/Parish`
from the general information file. (The hospital-wide measure, `Hybrid_HWR`, is in the file but every
score is `Not Available` in the current release. Check it; that is data quality in the wild.)

**Baseline:** the cell builds `hosp` with columns `Facility ID, Facility Name, State, County/Parish, readm_rate`.
Report how many hospitals have a rate, confirm `Facility ID` is unique in `hosp`, and confirm the
hospital-wide measure really is empty.

In [ ]:
uhv = load_cms("unplanned_visits")
hospitals = load_cms("hospital_general_info")
hf = uhv[uhv["Measure ID"].eq("READM_30_HF")][["Facility ID", "Score"]]
hf = hf.assign(readm_rate=to_num(hf["Score"])).dropna(subset=["readm_rate"]).drop(columns="Score")
hosp = hospitals[["Facility ID", "Facility Name", "State", "County/Parish"]].merge(hf, on="Facility ID", how="inner")
# TODO: how many hospitals have a rate? is Facility ID unique? how many Hybrid_HWR scores are numeric?

## 3. The join, and how much of it fails
PLACES has no county name in CMS's format, and CMS has no FIPS code. The only shared key is
*(state, county name)*, and names do not always agree (`ST. LOUIS` vs `St. Louis`, `DONA ANA` vs `Doña Ana`,
parishes, boroughs, and Connecticut, which now reports *planning regions* instead of counties).

**Baseline:** join `hosp` to `county` on upper-cased state and county name. Report the match rate
overall and the ten states with the worst match rate. Look at the unmatched names and fix at least
one systematic cause.

In [ ]:
# Naive join: upper-case the names and hope. Your job: measure how often it fails, find out why,
# and improve the normalization until the only failures left are ones no string fix can solve.
county["key_state"] = county["stateabbr"].str.upper()
county["key_county"] = county["locationname"].str.upper().str.strip()
hosp["key_state"] = hosp["State"].str.upper()
hosp["key_county"] = hosp["County/Parish"].str.upper().str.strip()
joined = hosp.merge(county, on=["key_state", "key_county"], how="left", indicator=True)
# TODO: match rate overall and by state; list the unmatched (State, County/Parish) pairs; fix a cause; re-join.

**What failed and why (edit this cell):**

## 4. One disparity, one chart
**Baseline:** split matched hospitals into quartiles of their county's uninsured rate (`ACCESS2`). For each
quartile, show the mean heart-failure readmission rate and the number of hospitals, and draw the chart.
Then write two sentences: what the chart shows, and one reason it is *not* evidence that lack of
insurance causes readmissions.

In [ ]:
m = joined[joined["_merge"].eq("both")].copy()
# TODO: pd.qcut on ACCESS2 into 4 bins; group; chart.

**Your two sentences:** *(edit this cell)*

## 5. Powerups
**(a) Census ACS.** Pull median household income by county from the Census API
(`https://api.census.gov/data/2022/acs/acs5`, variable `B19013_001E`). The API now **requires a free key**:
request one at https://api.census.gov/data/key_signup.html (arrives by email in minutes) and paste it
below. Join on FIPS (`locationid`), which is cleaner than names. Repeat section 4 with income.

**(b) SVI.** `load_svi_county()` in the setup cell points at the CDC/ATSDR Social Vulnerability Index.
Verify the URL on the ATSDR download page, load it, and use `RPL_THEMES` (overall vulnerability percentile)
as the x-axis instead of the uninsured rate. FIPS joins; no name matching.

**(c) County Health Rankings.** Same with `load_county_health_rankings()`; pick one measure from the data
dictionary and defend the choice.

In [ ]:
# TODO (powerup a): Census ACS pull. Paste your key; never commit it.
CENSUS_API_KEY = ""
ACS = "https://api.census.gov/data/2022/acs/acs5"

## Milestone — dataset documented (due with this lab)
In your capstone notebook: the dataset(s) you chose, a data dictionary (every column you will use, its
type, its unit, its source), and the `check()` function from week 3 adapted to it, passing.